In [1]:
import torch
from transformers import CLIPImageProcessor, CLIPVisionModel
import sys
sys.path.append("../face_anon_simple")

from diffusers import AutoencoderKL, DDPMScheduler
from diffusers.utils import load_image, make_image_grid
from src.diffusers.models.referencenet.referencenet_unet_2d_condition import (
    ReferenceNetModel,
)
from src.diffusers.models.referencenet.unet_2d_condition import UNet2DConditionModel
from src.diffusers.pipelines.referencenet.pipeline_referencenet import (
    StableDiffusionReferenceNetPipeline,
)

In [2]:
custom_cache_dir = "face_anon_simple/new_models"  # Change this to a directory with more space

face_model_id = "hkung/face-anon-simple"
clip_model_id = "openai/clip-vit-large-patch14"
sd_model_id = "stabilityai/stable-diffusion-2-1"

print("Start unet download")
unet = UNet2DConditionModel.from_pretrained(
    face_model_id, subfolder="unet", use_safetensors=True, cache_dir=custom_cache_dir
)
# state_dict = torch.load("/projectnb/cs585bp/projects/face_anonymization_proj/face_anon_simple/naman_ft_checkpoints_ce_200/epoch_19/unet.pt", map_location="cuda:0")   # <- your .pt file
# unet.load_state_dict(state_dict, strict=True)
print("Finished unet download")

print("Start Reference net download")
referencenet = ReferenceNetModel.from_pretrained(
    face_model_id, subfolder="referencenet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished reference net download")

print("Start condition reference net download")
conditioning_referencenet = ReferenceNetModel.from_pretrained(
    face_model_id, subfolder="conditioning_referencenet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished condition reference net download")

vae = AutoencoderKL.from_pretrained(
    sd_model_id, subfolder="vae", use_safetensors=True, cache_dir=custom_cache_dir
)
scheduler = DDPMScheduler.from_pretrained(
    sd_model_id, subfolder="scheduler", use_safetensors=True, cache_dir=custom_cache_dir
)
feature_extractor = CLIPImageProcessor.from_pretrained(
    clip_model_id, use_safetensors=True, cache_dir=custom_cache_dir
)
image_encoder = CLIPVisionModel.from_pretrained(
    clip_model_id, use_safetensors=True, cache_dir=custom_cache_dir
)

# pipe = StableDiffusionReferenceNetPipeline(
#     unet=unet,
#     referencenet=referencenet,
#     conditioning_referencenet=conditioning_referencenet,
#     vae=vae,
#     feature_extractor=feature_extractor,
#     image_encoder=image_encoder,
#     scheduler=scheduler,
# )

# pipe = pipe.to("cuda")

generator = torch.manual_seed(1)

Start unet download
Finished unet download
Start Reference net download


/projectnb/cs585bp/projects/face_anonymization_proj/.conda/face-anon-simple/lib/python3.8/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Finished reference net download
Start condition reference net download
Finished condition reference net download


In [ ]:
state_dict = torch.load("PATH OF UNET.PT", map_location="cuda:0")   # <- your .pt file
unet.load_state_dict(state_dict, strict=True)

pipe_ft = StableDiffusionReferenceNetPipeline(
    unet=unet,
    referencenet=referencenet,
    conditioning_referencenet=conditioning_referencenet,
    vae=vae,
    feature_extractor=feature_extractor,
    image_encoder=image_encoder,
    scheduler=scheduler,
)

In [4]:
from transformers import AutoImageProcessor, ViTForImageClassification
from PIL import Image
import torch
# Load the model and processor
custom_cache_dir_class="../classifier_gender/classifier_model"
model_name = "rizvandwiki/gender-classification-2"

classifier_model = ViTForImageClassification.from_pretrained(model_name, cache_dir=custom_cache_dir_class)
processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=custom_cache_dir_class)

# 'gender_dataset/CelebA_HQ_face_gender_dataset/train/male'
# image_path ="./my_dataset/train/celeb/real/01758_09704.png"
#[Failed for guy3]
# image = Image.open(image_path).convert("RGB") ""

In [5]:
import os
import gc
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import save_image
from PIL import Image
import gc 
from tqdm import tqdm, trange
import face_alignment
from utils.anonymize_faces_in_image import anonymize_faces_in_image
from torch.utils.data import Subset
from torch.utils.data import random_split
import numpy as np
import torch.nn.functional as F
import torchvision.transforms.functional as TF
# !pip install matplotlib
import matplotlib.pyplot as plt
from diffusers.utils import load_image
from collections import Counter
from collections import defaultdict
import random
from sklearn.model_selection import StratifiedShuffleSplit
from torch.cuda.amp import autocast, GradScaler

import bitsandbytes.optim as bnb_optim


import torch.nn as nn
import torchvision.models as models


In [ ]:
transform_raw = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])
transform_for_classifier = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])
dataset = ImageFolder(
    root="PATH OF TRAINING FOLDER OF FACE IMAGES",
    transform=transform_raw
)
print(dataset)
female_indices = range(2000)
male_indices = range(21000, 23000)
balanced_indices = list(female_indices) + list(male_indices)

dataset = Subset(dataset, balanced_indices)
train_len = int(0.8 * len(dataset))
val_len = int(0.1 * len(dataset))
test_len = len(dataset) - train_len - val_len
split_generator = torch.Generator().manual_seed(2)

train_set, val_set, test_set = random_split(
    dataset, [train_len, val_len, test_len], generator=split_generator
)


#Take the test set
batch_size = 1
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

pipe_ft.to("cuda:0")
classifier_model.to("cuda:0")

Dataset ImageFolder
    Number of datapoints: 23999
    Root location: /projectnb/cs585bp/projects/face_anonymization_proj/face_anon_simple/gender_dataset/CelebA_HQ_face_gender_dataset/train
    StandardTransform
Transform: Compose(
               Resize(size=(512, 512), interpolation=bilinear, max_size=None, antialias=warn)
               ToTensor()
           )


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=7

In [ ]:
import os
import torch
import numpy as np
from torchvision.utils import save_image
from torchvision import transforms
from tqdm import tqdm

# Create output folders
save_path = "PATH OF OUTPUT DIR"
os.makedirs(f"{save_path}/originals", exist_ok=True)
os.makedirs(f"{save_path}/anonymized", exist_ok=True)

correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for val_idx, (test_image_tensor, test_label) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        test_image = transforms.ToPILImage()(test_image_tensor[0]).convert("RGB")

        torch.manual_seed(1)
        np.random.seed(1)

        generated_test_tensor = pipe_ft(
            source_image=test_image,
            conditioning_image=test_image,
            num_inference_steps=10,
            guidance_scale=4.0,
            generator=generator,
            output_type='pt',  # already returns a tensor
            anonymization_degree=1.25,
        ).images[0]

        # Save the generated anonymized image


        # Classification
        processed_test_tensor = transform_for_classifier(generated_test_tensor)
        test_inputs = {"pixel_values": processed_test_tensor.unsqueeze(0).to("cuda:0")}
        test_logits = classifier_model(**test_inputs).logits
        test_pred = test_logits.argmax(dim=1).item()

        actual_label = test_label[0].item()

        if test_pred == actual_label:
            correct_predictions += 1
        total_predictions += 1
        anon_path = f"{save_path}/anonymized/anon_{val_idx}.png"
        save_image(generated_test_tensor.detach().cpu().clamp(0, 1), anon_path)

        # Save the original test image tensor
        original_tensor = test_image_tensor[0].detach().cpu().clamp(0, 1)
        original_path = f"{save_path}/originals/original_{val_idx}.png"
        save_image(original_tensor, original_path)

print(f"Correct Predictions: {correct_predictions}/{total_predictions}")
print(f"Accuracy: {correct_predictions / total_predictions:.2%}")



Evaluating:   0%|          | 0/400 [00:00<?, ?it/s]/projectnb/cs585bp/projects/face_anonymization_proj/.conda/face-anon-simple/lib/python3.8/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(
Evaluating: 100%|██████████| 400/400 [13:47<00:00,  2.07s/it]

Correct Predictions: 393/400
Accuracy: 98.25%
